# Notebook 04 — NumPy Fundamentals

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 04 of 20  
**Prerequisites:** Notebook 03  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Gene expression data arrives as matrices: rows are genes (20,000+), columns are samples
(6–1000+). Looping over Python lists to compute means, standard deviations, or
correlations on such matrices would take minutes. NumPy does the same in milliseconds
by storing data in contiguous memory and delegating to compiled C/Fortran kernels.

This notebook establishes the array mental model and the core operations used in
every later module.

---
## Step 2 — Intuition

An `ndarray` is a rectangular block of memory with a shape and a dtype.
The shape tells you the dimensions; the dtype tells you how many bytes each element
occupies. All elements share the same dtype — that's why arithmetic is fast.

Think of a 2-D array as a spreadsheet where every cell is the same type,
and the entire spreadsheet lives in one contiguous block of RAM.

---
## Step 3 — Biological Background

**Gene expression matrix:**

```
            Sample_1  Sample_2  Sample_3
BRCA1       5.2       3.1       8.9
TP53        12.1      11.8      13.2
MYC         2.3       4.5       1.9
```

This is a NumPy array `X` of shape `(n_genes, n_samples)`.  
- `X[i, j]` = expression of gene $i$ in sample $j$  
- `X.mean(axis=1)` = mean expression of each gene across all samples  
- `X.mean(axis=0)` = mean expression per sample across all genes  

Axis arithmetic is the most frequent source of bugs in expression analysis.

---
## Step 4 — Mathematical Explanation

Let $X \in \mathbb{R}^{m \times n}$ (genes × samples).  

**Column-wise z-score normalization:**
$$\hat{X}_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}$$

where $\mu_j = \frac{1}{m}\sum_i X_{ij}$ and $\sigma_j = \sqrt{\frac{1}{m}\sum_i (X_{ij}-\mu_j)^2}$.

After this transform, each sample column has mean 0 and standard deviation 1 — a
prerequisite for many downstream analyses including PCA and clustering.

---
## Step 5 — Computational Explanation

**Core NumPy concepts:**

| Concept | Description |
|---------|-------------|
| `ndarray` | N-dimensional array of fixed dtype |
| Shape / reshape | The dimensions of an array; can be changed without copying |
| Indexing / slicing | Element, row, column, boolean, fancy indexing |
| Universal functions (ufuncs) | Element-wise operations (`np.sqrt`, `np.log`, `np.exp`) |
| Reduction functions | Aggregate along an axis (`mean`, `std`, `sum`, `min`, `max`) |
| Broadcasting | Arithmetic between arrays of different shapes — Step 5 of NB05 |
| Views vs copies | Slicing returns a view; copying requires `.copy()` |

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
print(f"NumPy {np.__version__}")

In [ ]:
# Cell 6.1 — Creating arrays
# Synthetic expression data: 5 genes, 3 samples
rng = np.random.default_rng(seed=42)
X = rng.exponential(scale=5.0, size=(5, 3)).round(2)

gene_names = ["BRCA1", "TP53", "MYC", "EGFR", "KRAS"]
sample_names = ["Control", "Treated_1", "Treated_2"]

print(f"Shape: {X.shape}   dtype: {X.dtype}")
print(f"Memory footprint: {X.nbytes} bytes")
print()
print(f"{'Gene':<10}", "  ".join(f"{s:<10}" for s in sample_names))
print("-" * 40)
for name, row in zip(gene_names, X):
    print(f"{name:<10}", "  ".join(f"{v:<10.2f}" for v in row))

In [ ]:
# Cell 6.2 — Indexing
print("First gene, all samples:     ", X[0, :])    # row
print("All genes, first sample:     ", X[:, 0])    # column
print("TP53 in Treated_1:           ", X[1, 1])    # scalar
print("Submatrix (genes 1-3, s 1-2):\n", X[1:4, 1:3])

# Boolean indexing: genes with mean expression > 4
high_expr_mask = X.mean(axis=1) > 4
print("\nHigh-expression genes (mean > 4):")
for name, flag, mean in zip(gene_names, high_expr_mask, X.mean(axis=1)):
    print(f"  {name:<8}  mean={mean:.2f}  high={flag}")

In [ ]:
# Cell 6.3 — Reductions along axes
print("Mean per gene (axis=1):", X.mean(axis=1).round(3))    # shape (5,)
print("Mean per sample (axis=0):", X.mean(axis=0).round(3))  # shape (3,)
print("Overall mean:", X.mean().round(3))                     # scalar
print()
print("Std per gene:", X.std(axis=1).round(3))
print("Max per gene:", X.max(axis=1).round(3))

# argmax: which sample has the highest expression for each gene?
peak_sample = X.argmax(axis=1)
print("\nPeak sample index per gene:", peak_sample)
print("Peak sample name per gene: ", [sample_names[i] for i in peak_sample])

In [ ]:
# Cell 6.4 — z-score normalization (per sample = per column)
def zscore_columns(X: np.ndarray) -> np.ndarray:
    """Z-score normalize each column; return array of same shape."""
    mu = X.mean(axis=0)          # shape (n_samples,)
    sigma = X.std(axis=0)        # shape (n_samples,)
    sigma[sigma == 0] = 1.0      # avoid division by zero
    return (X - mu) / sigma      # broadcasting: (n_genes, n_samples)

Z = zscore_columns(X)
print("Z-scored matrix:")
print(Z.round(3))
print()
print("Column means (should be ~0):", Z.mean(axis=0).round(10))
print("Column stds  (should be ~1):", Z.std(axis=0).round(3))

In [ ]:
# Cell 6.5 — Views vs copies
row_view = X[0]          # This is a VIEW — modifying it changes X!
row_copy = X[0].copy()   # This is a COPY — safe to modify

original_val = X[0, 0]
row_view[0] = 9999.0
print(f"After modifying view: X[0,0] = {X[0,0]}")   # changed!

X[0, 0] = original_val   # restore
row_copy[0] = 9999.0
print(f"After modifying copy: X[0,0] = {X[0,0]}")   # unchanged

print("\nRule: index slices return views; boolean/fancy indexing returns copies.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Raw vs z-scored expression heatmap
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data, title in zip(axes, [X, Z], ["Raw expression", "Z-scored (per sample)"]):
    im = ax.imshow(data, aspect="auto", cmap="RdBu_r")
    ax.set_xticks(range(len(sample_names)))
    ax.set_xticklabels(sample_names, rotation=20, ha="right")
    ax.set_yticks(range(len(gene_names)))
    ax.set_yticklabels(gene_names)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Expression matrix — raw vs normalized", fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Write a function `log2_normalize(X, pseudocount=1)` that returns `log2(X + pseudocount)`.
   Explain why the pseudocount is necessary.
2. Generate a 1000×20 random expression matrix and measure (with `time.perf_counter`) the
   time to z-score it with NumPy vs a pure Python double-loop. Report the speedup.
3. Write `top_k_genes(X, k)` that returns the `k` row indices with the highest mean
   expression. Use `np.argsort`.
4. Explain in one sentence why `X[0]` returns a view but `X[mask]` (boolean indexing)
   returns a copy.

---
## Quiz — Active Recall

1. What does `X.mean(axis=0)` compute for a genes×samples matrix? What shape does it return?
2. What is the dtype of a NumPy array created from `[1, 2, 3]`? From `[1.0, 2, 3]`?
3. If `X` has shape `(100, 5)`, what shape does `X - X.mean(axis=0)` produce? Why?
4. Why is `sigma[sigma == 0] = 1.0` in `zscore_columns` preferable to catching a ZeroDivisionError?
5. What is a ufunc? Give two examples used in expression analysis.

---
## Reflection

**Date completed:** ____________________

> *[Could you implement zscore_columns from scratch with the notebook closed? What was surprising about views vs copies? Which exercise took longest?]*

---
*Next: `05_broadcasting_vectorization.ipynb`*